# Single-Cell Metadata Preprocessing Demo

**Author:** Yiran Song  
**Date:** May 10, 2025

This document demonstrates how to preprocess single cell metadata using
the GSE195673 dataset as an example.  
Please note that each dataset may have unique characteristics, so this
example is intended to illustrate a general approach.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import pandas as pd
# pd.set_option('display.max_columns', None)

# GSE195673
GSE195673 = ad.read_h5ad("./h5ad_files/GSE195673_WU368.h5ad")
print(GSE195673.obs.head())
GSE195673

In [ ]:
GSE195673.obs_names = [f"{idx}_GSE195673" for idx in GSE195673.obs_names]
GSE195673.obs_names

In [ ]:
import re
def extract_time_value(s, pattern):
    """Helper to extract numerical time values from strings using regex."""
    match = re.search(pattern, str(s))
    return float(match.group(1)) if match else np.nan

def parse_gse195673_time(x):
    if pd.isna(x):
        return np.nan
    if "+" in str(x):  # Handle "d28+d35" case
        days = [float(d[1:]) for d in x.split("+")]
        return sum(days) / len(days)  # Average of the days
    return float(x[1:])  # Extract number after "d"

# Apply to GSE195673
GSE195673.obs["time_point_day"] = GSE195673.obs["timepoint"].apply(parse_gse195673_time)
GSE195673.obs["time_point_day"] = GSE195673.obs["time_point_day"].astype(float)
GSE195673.obs["time_point_hour"] = GSE195673.obs["time_point_day"] * 24
GSE195673.obs["time_point_week"] = GSE195673.obs["time_point_day"] / 7
print(GSE195673.obs['time_point_hour'].unique())
print(GSE195673.obs['time_point_day'].unique())
print(GSE195673.obs['timepoint'].unique())

GSE195673.obs['study_type'] = 'vaccine'

GSE195673.obs.rename(columns={
    'donor': 'donor_id',
    'sample': 'sample',
    'anno_leiden_0.23': 'cell_type',
    'biosample': 'sample_id',
    'sorting': 'tissue'
}, inplace=True)
GSE195673

In [ ]:
GSE195673.obs['dataset'] = 'GSE195673'
print(GSE195673.obs['donor_id'].unique())
print(GSE195673.obs['sample_id'].unique())
print(GSE195673.obs['dataset'].unique())
print(GSE195673.obs['cell_type'].unique())
print(GSE195673.obs['study_type'].unique())
print(GSE195673.obs['tissue'])
# GSE195673.write("./meta_processed/GSE195673.h5ad")